Functions to: (1) Translate structured criteria list to an SQL query handling joins (2) Basic validation of each condition (table.field exists in DB, value type compatibility)

In [ ]:
#https://docs.sqlalchemy.org/en/20/intro.html#installation
#!pip install SQLAlchemy

In [58]:
from sqlalchemy import Table, Column, Integer, String, MetaData, select, or_, and_, not_, case, func
from sqlalchemy.sql import text
import ast
import json
import collections
from collections import deque
from rich import print

In [ ]:
# OpenAI API Key (only needed for the string validation function later)
from openai import OpenAI
import os

os.environ['OPENAI_API_KEY'] = "sk-..."
client = OpenAI()

In [ ]:
# --- Helper functions to create the joins needed to combine data across tables ---

def find_join_path(start_table, end_table, schema_keys):
    """
    Finds the shortest sequence of tables to join from start to end
    using a schema definition with primary and foreign keys.
    """
    if start_table == end_table:
        return [start_table]

    queue = deque([[start_table]])
    visited = {start_table}

    while queue:
        path = queue.popleft()
        current_table = path[-1]

        # --- Find all neighbors using the PK/FK schema ---
        neighbors = set()

        # 1. Find tables that `current_table` points to (via its own FKs)
        for referenced_table in schema_keys[current_table]['fks'].values():
            neighbors.add(referenced_table)

        # 2. Find tables that point to `current_table` (reverse lookup)
        for other_table, details in schema_keys.items():
            if current_table in details['fks'].values():
                neighbors.add(other_table)

        # Process the found neighbors
        for neighbor in neighbors:
            if neighbor not in visited:
                new_path = list(path)
                new_path.append(neighbor)
                if neighbor == end_table:
                    return new_path  # Path found

                visited.add(neighbor)
                queue.append(new_path)

    return None # No path exists

def get_join_keys(table1_name, table2_name, schema_keys):
    """
    Finds the foreign key and primary key to join two adjacent tables.
    """
    t1_details = schema_keys[table1_name]
    t2_details = schema_keys[table2_name]

    # Case 1: Table 1 has an FK pointing to Table 2's PK
    for fk_col, ref_table in t1_details['fks'].items():
        if ref_table == table2_name:
            # Join on table1.fk_col = table2.pk
            return (fk_col, t2_details['pk'])

    # Case 2: Table 2 has an FK pointing to Table 1's PK
    for fk_col, ref_table in t2_details['fks'].items():
        if ref_table == table1_name:
            # Join on table1.pk = table2.fk_col
            return (t1_details['pk'], fk_col)
            
    return None # Should not happen if a path was found

In [51]:
# --- SQL query builder ---
def build_query_from_criteria(criteria, tables, schema_keys, root="main"):
    """
    Builds an SQL query with ORM functions, handling complex JOINs and AND/OR in condition strings 
    and multi-entry conditions via GROUP BY/HAVING.
    """
    
    # 1. Condition parser
    def _build_expression(condition_str):
        """Recursively builds a SQLAlchemy expression from a string."""
        # Base case of recursion: a simple 'table.col op value' clause
        def _parse_simple_clause(clause_str):
            parts = clause_str.strip().split(maxsplit=2)
            table, col = parts[0].split(".")
            op, val_str = parts[1], parts[2]  
            
            col_expr = tables[table].c[col]

            # Handle IN clause
            if op.upper() == 'IN':
                try:
                    # Safely evaluate the string representation of the list
                    list_of_values = ast.literal_eval(val_str)
                    if not isinstance(list_of_values, list):
                        raise ValueError("Value for IN operator must be a list.")
                    return col_expr.in_(list_of_values)
                except (ValueError, SyntaxError) as e:
                    raise ValueError(f"Could not parse list for IN condition: {val_str}") from e

            # Handle other operators
            else:
                # Strip quotes if string, else cast to int/float
                if val_str.startswith("'") and val_str.endswith("'"): 
                    val = val_str.strip("'")
                else:
                    try: val = int(val_str)
                    except ValueError: val = float(val_str)

                # Operator mapping
                ops = { "=": lambda c, v: c == v, ">": lambda c, v: c > v, "<": lambda c, v: c < v,
                        ">=": lambda c, v: c >= v, "<=": lambda c, v: c <= v, "!=": lambda c, v: c != v }
                if op not in ops:
                    raise ValueError(f"Unsupported operator: {op}")
            return ops[op](col_expr, val)

        # Recursive step: split by OR, then by AND
        or_parts = [p.strip() for p in condition_str.split(' OR ')]
        or_clauses = []
        for part in or_parts:
            and_parts = [p.strip() for p in part.split(' AND ')]
            and_clauses = [_parse_simple_clause(p) for p in and_parts]
            or_clauses.append(and_(*and_clauses))
            print(part, and_parts)
        
        return or_(*or_clauses)

    # 2. Parse and categorize criteria (WHERE v/s GROUP_BY/HAVING)
    include_counts = collections.Counter()
    for c in criteria:
        if c['type'] == 'include':
            # This simple check is sufficient to find multi-entry cases
            simple_cond = c['condition'].split(' AND ')[0].split(' OR ')[0]
            table, col = simple_cond.strip().split(maxsplit=2)[0].split('.')
            include_counts[(table, col)] += 1
    
    where_filters = []
    having_filters = []
    for c in criteria:
        simple_cond = c['condition'].split(' AND ')[0].split(' OR ')[0]
        table, col = simple_cond.strip().split(maxsplit=2)[0].split('.')
        is_multi_entry = include_counts.get((table, col), 0) > 1

        # A condition is for HAVING only if it's simple and part of a multi-entry set
        is_simple_condition = ' AND ' not in c['condition'] and ' OR ' not in c['condition']
        if c['type'] == 'include' and is_multi_entry and is_simple_condition:
            having_filters.append(c)
        else:
            where_filters.append(c)
    print(f"Where:\n{where_filters}")
    print(f"Having:\n{having_filters}")

    # 3. Initialize query, build joins across reqd tables, handle joins across tables not sharing PK/FK with bridging
    root_table = tables[root]
    root_pk = schema_keys[root]['pk']
    grouping_key = root_table.c[root_pk]   # assume the pk of root table is the reference key for grouping
    # initialize query obj
    query = select(grouping_key) # TO-DO: provide option to supply more keys for data fetching
    
    # Get the tables from the criteria
    required_tables = set()
    for c in criteria:
        # extract table names from table.field mentions in each condition
        words = c["condition"].split()
        for word in words:
            if '.' in word:
                table_name = word.split('.')[0]
                if table_name in tables:
                    required_tables.add(table_name)
    print(f'Tables in criteria: {required_tables}')

    # Find the join path for each required table and add necessary joins
    tables_in_query = {root}
    for target_table in required_tables:
        if target_table in tables_in_query:
            continue

        # Call the new pathfinder
        path = find_join_path(root, target_table, schema_keys)
        if path is None:
            raise Exception(f"No join path found from '{root}' to '{target_table}'")

        # Iterate through the path to build joins
        for i in range(len(path) - 1):
            from_table_name = path[i]
            to_table_name = path[i+1]
            if to_table_name not in tables_in_query:
                # Get the join columns using the new helper
                from_key, to_key = get_join_keys(from_table_name, to_table_name, schema_keys)

                from_table = tables[from_table_name]
                to_table = tables[to_table_name]

                query = query.join(to_table, from_table.c[from_key] == to_table.c[to_key])
                tables_in_query.add(to_table_name)
    print(f'Tables in the framed query: {tables_in_query}') # this can include bridging tables

    # 4. Apply row-level filters (WHERE caluse)
    for f in where_filters:
        expression = _build_expression(f['condition'])
        query = query.filter(expression if f['type'] == 'include' else not_(expression))

    # 5. APPLY group-level filters (HAVING clause)
    if having_filters:
        query = query.group_by(grouping_key)
        having_clauses = []
        for f in having_filters:
            # Having filters are guaranteed to be simple by our categorization logic
            parts = f['condition'].strip().split(maxsplit=2)
            table, col = parts[0].split('.'); val_str = parts[2]
            if val_str.startswith("'"): val = val_str.strip("'")
            else: val = int(val_str)
            col_expr = tables[table].c[col]
            condition = func.count(case((col_expr == val, 1))) > 0
            having_clauses.append(condition)
        query = query.having(and_(*having_clauses))

    return str(query.compile(compile_kwargs={"literal_binds": True}))

In [39]:
## Initialize the objects needed to be passed to sql function

# dummy schema as key/value pairs
schema = {
    "main": ["curated_patient_id", "kw_curated_drug", "diabetes_treatment_type", "disease"],
    "Diagnosis": ["curated_patient_id", "cid", "primary_diagnosis"],
    "Donor": ["id","cid", "gender", "bmi", "weight_in_kg"]
}

# Define a dict with pk/fk mappings per table
schema_keys = {
    'main': {
        'pk': 'curated_patient_id',
        'fks': {}
    },
    'Diagnosis': {
        'pk': 'cid',
        'fks': {
            'curated_patient_id': 'main',
        }
    },
    'Donor': {
        'pk': 'id',
        'fks': {
            'cid': 'Diagnosis',
        }
    }
}

# Tables in SQAlchemy compatible format
metadata = MetaData()
tables = {
    t: Table(t, metadata, *(Column(c, String) for c in cols)) 
    for t, cols in schema.items()
}
print(len(tables))

3

In [40]:
# Example of join across tables not sharing pk/fk
start = "main" 
end = "Donor" 

join_keys = get_join_keys(start, end, schema_keys)
print(f"Join keys between {start} and {end}: {join_keys}")

found_path = find_join_path(start, end, schema_keys) 
print(f"Path from '{start}' to '{end}': {found_path}")

Join keys between main and Donor: None

Path from 'main' to 'Donor': ['main', 'Diagnosis', 'Donor']

In [49]:
# Test dummy criteria
criteria = [
        {"type": "include", "condition": "Donor.gender = 'female'"},
        {"type": "exclude", "condition": "Donor.bmi > 20 OR Donor.weight_in_kg > 90"},
        {"type": "include", "condition": "main.kw_curated_drug = 'heparin'"},
        {"type": "include", "condition": "main.kw_curated_drug = 'insulin'"},
        {"type": "include", "condition": "main.disease IN ['IC','UC']"},
    ]

In [50]:
crit_sql = build_query_from_criteria(criteria, tables, schema_keys, root="main")
print("Translated Query:\n", crit_sql)

Where:
[{'type': 'include', 'condition': "Donor.gender = 'female'"}, {'type': 'exclude', 'condition': 'Donor.bmi > 20 OR 
Donor.weight_in_kg > 90'}, {'type': 'include', 'condition': "main.disease IN ['IC','UC']"}]

Having:
[{'type': 'include', 'condition': "main.kw_curated_drug = 'heparin'"}, {'type': 'include', 'condition': 
"main.kw_curated_drug = 'insulin'"}]

Tables in criteria: {'main', 'Donor'}

Tables in the framed query: {'main', 'Diagnosis', 'Donor'}

Donor.gender = 'female'
["Donor.gender = 'female'"]

Donor.bmi > 20
['Donor.bmi > 20']

Donor.weight_in_kg > 90
['Donor.weight_in_kg > 90']

main.disease IN ['IC','UC']
["main.disease IN ['IC','UC']"]

Translated Query:
 SELECT main.curated_patient_id 
FROM main JOIN "Diagnosis" ON main.curated_patient_id = "Diagnosis".curated_patient_id JOIN "Donor" ON 
"Diagnosis".cid = "Donor".cid 
WHERE "Donor".gender = 'female' AND NOT ("Donor".bmi > 20 OR "Donor".weight_in_kg > 90) AND main.disease IN ('IC', 
'UC') GROUP BY main.curated_patient_id 
HAVING count(CASE WHEN (main.kw_curated_drug = 'heparin') THEN 1 END) > 0 AND count(CASE WHEN (main.kw_curated_drug
= 'insulin') THEN 1 END) > 0

In [ ]:
# --- Helper functions to do basic validation checks on individual conditions ---

from typing import List, Dict, Any, Literal, Tuple

def _llm_json(prompt: str, schema: Dict[str, Any]) -> Any:
    """Call LLM and expect JSON output matching schema."""
    resp = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature = 0.0,
    messages=[{"role": "user", "content": prompt}],
    response_format={"type": "json_schema", "json_schema": schema},
    )
    return json.loads(resp.choices[0].message.content)

def _validate_single_value(value, col_type, col_info):
    """Helper function to validate one value against the column schema."""
    val_type = "str"
    # Determine the type of the value (not a string representation)
    if isinstance(value, int):
        val_type = "int"
    elif isinstance(value, float):
        val_type = "float"

    NUMERIC_TYPES = {'int64', 'float64', 'int', 'float'}
    is_val_numeric = val_type in NUMERIC_TYPES
    is_col_numeric = col_type in NUMERIC_TYPES

    # if both val and col are numeric
    if is_val_numeric and is_col_numeric:
        return {"valid": True, "reason": "Numeric value is compatible with numeric column."}
    
    # type mismatch
    if is_val_numeric != is_col_numeric:
        return {"valid": False, "reason": f"Type Mismatch: Value {value} ({val_type}) is incompatible with column type '{col_type}'."}

    # Both are non-numeric, escalate to LLM for format check
    # Note: This is WIP and may give some false positives
    prompt = f"""
        You are a format validator.
        Check if the filtering value's string format is compatible with the column's string format.
        Do not expect a specific match with the provided sample column values, only check for pattern.
        Filtering Value: {value}
        Column Summary: {col_info}
        Result (JSON):
    """
    schema = {
        "name": "ValidatedExpr",
        "schema": {
            "type": "object",
            "properties": {
                "valid": {"type": "boolean"},
                "reason": {"type": "string", "description": "short reason"},
            },
            "required": ["valid", "reason"]
        }    
    }
    response = _llm_json(prompt, schema)
    return response

def _validate_expr(expr, db_schema):
    """
    Validates an expression, handling standard operators and IN clauses with list value.
    Expression format: '<table.field> <operator> <value>'.
    """
    # 1. Parse the expression
    try:
        field_part, op, val_str = expr.strip().split(maxsplit=2)
        table, col = field_part.split(".")
    except ValueError:
        return {"valid": False, "reason": f"Invalid expression format. Expected '<table.field> <op> <value>'."}

    # 2. Get column info from DB schema
    try:
        col_info = db_schema[table]['fields'][col]
        col_type = col_info["field_data_type"].lower()
    except KeyError:
        return {"valid": False, "reason": f"Column '{table}.{col}' not found in database schema."}

    op = op.upper() # Standardize operator to uppercase

    # 3. Handle IN clause
    if op == 'IN':
        try:
            # Safely evaluate the string as a Python literal (e.g., "['a', 'b']")
            value_list = ast.literal_eval(val_str)
            if not isinstance(value_list, list):
                raise TypeError
        except (ValueError, SyntaxError, TypeError):
            return {"valid": False, "reason": "Operator 'IN' must be followed by a valid list (e.g., ['a', 'b'])."}
        
        if not value_list: # Empty list is valid
            return {"valid": True, "reason": "Expression with empty IN list is valid."}

        # Validate each item in the list
        in_results = []
        for item in value_list:
            result = _validate_single_value(item, col_type, col_info)
            if not result["valid"]:
                # Prepend context to the reason from the helper function
                result["reason"] = f"Invalid item in IN list. {result['reason']}"
                #return result
            in_results.append(result)
        if any(res["valid"] is False for res in in_results):
            return in_results
        else:
            return {"valid": True, "reason": "All items in the IN list are valid."}

    # 4. Handle other operators
    else:
        val = val_str.strip("'\"")
        parsed_val = val
        try:
            parsed_val = int(val)
        except ValueError:
            try:
                parsed_val = float(val)
            except ValueError:
                pass # Stays a str
        return _validate_single_value(parsed_val, col_type, col_info)

def validate_criteria(criteria_list: List[Dict[str, Any]], db_schema) -> List[Tuple]:
    """
    Validate value type (and format for strs) compatibility with the db column for each expression in the list of criteria.
    Also checks if each table.col in criteria is present in the DB.
    Return a list of tuples (single_condition, validation_flag).
    TO-DO: Parallelize checks and LLM calls.
    """

    validation_status = []
    for c in criteria_list:
        crit = c["condition"]
        # get single criteria by splitting on `OR` or `AND` clause
        if 'OR' in crit:
            exprs = crit.strip().split('OR')
            for expr in exprs:
                flag = _validate_expr(expr, db_schema)
                validation_status.append((expr.strip(), flag))
        elif 'AND' in crit:
            exprs = crit.strip().split('AND')
            for expr in exprs:
                flag = _validate_expr(expr, db_schema)
                validation_status.append((expr.strip(), flag))
        else:
            expr = crit.strip()
            flag = _validate_expr(expr, db_schema)
            validation_status.append((expr.strip(), flag))

    return validation_status

In [ ]:
# Loading the schema JSON needed for the above validation function
with open('./CPTAC_schema_v2.json', 'r') as f:
    DB_SCHEMA = json.load(f)

In [ ]:
# Run on the same dummy example from above, checking each single condition
validation_status = validate_criteria(criteria, DB_SCHEMA)

print("Criteria:", criteria)
for c in validation_status:
    print(c[0], c[1])

Criteria:
[
    {'type': 'include', 'condition': "Donor.gender = 'female'"},
    {'type': 'exclude', 'condition': 'Donor.bmi > 20 OR Donor.weight_in_kg > 90'},
    {'type': 'include', 'condition': "main.kw_curated_drug = 'heparin'"},
    {'type': 'include', 'condition': "main.kw_curated_drug = 'insulin'"},
    {'type': 'include', 'condition': "main.disease IN ['IC','UC']"}
]

Donor.gender = 'female'
{
    'valid': True,
    'reason': "The filtering value 'female' matches the expected string format for the gender column, which accepts
'male' and 'female'."
}

Donor.bmi > 20
{'valid': True, 'reason': 'Numeric value is compatible with numeric column.'}

Donor.weight_in_kg > 90
{'valid': True, 'reason': 'Numeric value is compatible with numeric column.'}

main.kw_curated_drug = 'heparin'
{
    'valid': False,
    'reason': "The filtering value 'heparin' does not match the expected pattern of the column, which primarily 
contains 'none'."
}

main.kw_curated_drug = 'insulin'
{
    'valid': False,
    'reason': "The filtering value 'insulin' does not match the expected format of the column, which primarily 
contains the value 'none'."
}

main.disease IN ['IC','UC']
{'valid': False, 'reason': "Column 'main.disease' not found in database schema."}